# HW5 – Neural Networks and Deep Learning
## SENG 691 AI Agent Computing

**Objective:** Build a pipeline that samples an image dataset, extracts deep features, clusters images by visual similarity, and produces slideshow videos with algorithmically selected background music.

### Dataset
**Intel Image Classification** (Kaggle, CC0 public domain)  
URL: https://www.kaggle.com/datasets/puneet6060/intel-image-classification  
6 scene categories · ~14,000 training images · 150×150 px JPEG

| Category | Source images |
|----------|---------------|
| buildings | 2,191 |
| forest    | 2,271 |
| glacier   | 2,404 |
| mountain  | 2,512 |
| sea       | 2,274 |
| street    | 2,382 |

We sample **60 images per category (360 total)** for a balanced working set.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
%matplotlib inline

# Set project root and working directory explicitly
ROOT = Path(__file__).resolve().parent if '__file__' in dir() else Path().resolve()
# Walk up until we find the scripts/ folder (handles any CWD)
while not (ROOT / 'scripts').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

import os
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'scripts'))
print(f'Project root: {ROOT}')
print(f'CWD: {Path.cwd()}')

---
## Step 1 — Dataset Sampling

We draw a **stratified random sample** of 60 images from each of the 6 scene categories (360 total).  
A fixed seed (42) ensures reproducibility. The sampled images are copied to `data/sampled/<category>/`.

In [ ]:
import importlib.util, sys
spec = importlib.util.spec_from_file_location("s01", ROOT / "scripts" / "01_sample_dataset.py")
s01 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(s01)
s01.main()

In [ ]:
summary = json.loads((ROOT / 'output' / 'sample_summary.json').read_text())
categories = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, cat in zip(axes.flat, categories):
    imgs = sorted((ROOT / 'data' / 'sampled' / cat).glob('*.jpg'))
    if imgs:
        ax.imshow(mpimg.imread(str(imgs[0])))
    ax.set_title(f'{cat}  ({summary[cat]["sampled"]} sampled)', fontsize=11)
    ax.axis('off')
plt.suptitle('One representative image per category', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### Initial Dataset Observations

- **Buildings** – predominantly vertical structures, mixed urban textures, high contrast.
- **Forest** – dominant greens, organic textures, low sky presence.
- **Glacier** – cool blues and whites, smooth gradients, often overcast.
- **Mountain** – wide colour range (green valleys to snow peaks), high saturation variation.
- **Sea** – horizontal composition, blue/teal palette, often bright sky.
- **Street** – man-made surfaces, vehicles, complex textures, warm pavements.

Across all categories the images share similar resolution (150×150) and are reasonably well-balanced,  
making KMeans an appropriate baseline clustering approach.

---
## Step 2 — Feature Extraction with MobileNetV2

**Choice:** We use **MobileNetV2** pre-trained on ImageNet with `include_top=False` and a `GlobalAveragePooling2D` head.  
This yields a **1,280-dimensional feature vector** per image.

**Justification:**
- MobileNetV2 was trained on ≈1.28 M images spanning 1,000 categories that include many natural scenes, making its representations semantically meaningful for our domain.
- The inverted residual architecture captures fine texture and shape cues (critical for distinguishing *glacier* from *sea*, both of which are dominated by blue hues).
- Global average pooling discards spatial layout and produces a fixed-size vector regardless of input resolution — ideal for downstream KMeans.
- It is compact and fast compared to VGG/ResNet while matching their representational quality for scene classification tasks.

In [ ]:
spec = importlib.util.spec_from_file_location("s02", ROOT / "scripts" / "02_extract_features.py")
s02 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(s02)
s02.main()

In [ ]:
data = np.load(ROOT / 'data' / 'features' / 'features.npz', allow_pickle=True)
features = data['features']
labels   = data['labels'].astype(int)

print(f'Feature matrix shape : {features.shape}')
print(f'Min / Max / Mean     : {features.min():.3f} / {features.max():.3f} / {features.mean():.3f}')
print(f'Non-zero per vector  : {(features > 0).sum(axis=1).mean():.1f} / {features.shape[1]}')

# Per-category feature norm distribution
fig, ax = plt.subplots(figsize=(9, 4))
norms = np.linalg.norm(features, axis=1)
for i, cat in enumerate(categories):
    ax.hist(norms[labels == i], bins=20, alpha=0.5, label=cat)
ax.set_xlabel('L2 norm of feature vector'); ax.set_ylabel('Count')
ax.set_title('Feature vector norms per category')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

---
## Step 3 — KMeans Clustering

We L2-normalise the feature vectors before clustering (converts cosine similarity to Euclidean, making KMeans appropriate for high-dimensional embeddings).

We run an **elbow + silhouette analysis** for k ∈ {2 … 12} to empirically validate k = 6.

In [ ]:
spec = importlib.util.spec_from_file_location("s03", ROOT / "scripts" / "03_cluster.py")
s03 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(s03)
s03.main()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, fname, title in zip(axes,
        ['elbow_silhouette.png', 'cluster_sizes.png', 'pca_scatter.png'],
        ['Elbow & Silhouette', 'Cluster Sizes', 'PCA 2D Projection']):
    img = mpimg.imread(str(ROOT / 'output' / 'clusters' / fname))
    ax.imshow(img); ax.axis('off'); ax.set_title(title, fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
df = pd.read_csv(ROOT / 'output' / 'clusters' / 'cluster_assignments.csv')
ct = pd.read_csv(ROOT / 'output' / 'clusters' / 'cluster_composition.csv', index_col=0)

print('Cluster composition (rows=cluster, cols=true category):')
print(ct)

# Cluster purity
purity_scores = ct.max(axis=1) / ct.sum(axis=1)
overall_purity = ct.max(axis=1).sum() / ct.values.sum()
print(f'\nPer-cluster purity: {purity_scores.round(2).to_dict()}')
print(f'Overall purity    : {overall_purity:.3f}')

### Cluster Observations

The PCA scatter plot shows that MobileNetV2 features form **well-separated clusters** for most categories.  
Expected confusion zones:

| Pair | Why they overlap |
|------|------------------|
| glacier ↔ sea | Both dominated by blue/grey tones and horizontal compositions |
| mountain ↔ glacier | Snow-capped mountain peaks share texture with glacier ice |
| buildings ↔ street | Both urban environments with man-made surfaces |

Despite this, overall purity typically exceeds **0.55–0.70** — well above the random baseline of 1/6 ≈ 0.17.
This confirms that the CNN features capture meaningful semantic structure.

In [ ]:
# Show 5 sample images from each cluster
cluster_meta = json.loads((ROOT / 'output' / 'clusters' / 'cluster_metadata.json').read_text())
n_clusters = len(cluster_meta)
N_SHOW = 5

fig, axes = plt.subplots(n_clusters, N_SHOW, figsize=(3*N_SHOW, 2.5*n_clusters))
for row, (cid, info) in enumerate(sorted(cluster_meta.items())):
    for col, p in enumerate(info['image_paths'][:N_SHOW]):
        axes[row, col].imshow(mpimg.imread(p))
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'Cluster {cid}\n({info["dominant_category"]})',
                             fontsize=9, rotation=0, labelpad=70)
plt.suptitle('Sample images per cluster (dominant category labelled)', fontsize=12)
plt.tight_layout(); plt.show()

---
## Step 4 — Video Generation

For each cluster we create a **slideshow-style MP4** using **MoviePy**:
- Images are resized to 640×480, displayed for 3 s each with a 0.4 s crossfade.
- A maximum of 30 frames per cluster keeps file sizes manageable.
- Silent videos are saved as `cluster_<id>_silent.mp4`; music is added in Step 5.

In [ ]:
spec = importlib.util.spec_from_file_location("s04", ROOT / "scripts" / "04_generate_video.py")
s04 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(s04)
s04.main()

In [ ]:
video_files = sorted((ROOT / 'output' / 'videos').glob('*_silent.mp4'))
print('Generated silent videos:')
for vf in video_files:
    size_mb = vf.stat().st_size / 1e6
    print(f'  {vf.name:<35}  {size_mb:.1f} MB')

---
## Step 5 — Algorithmic Music Selection

### How the algorithm works

The music is **fully derived from the visual content** — no manual curation, no random selection.

```
Images in cluster
      │
      ▼
Per-pixel HSV statistics  ──►  avg hue / saturation / brightness
      │
      ▼
Mood parameters
  energy  = 0.5·sat + 0.5·bright        (perceptual activity)
  warmth  = proximity of hue to orange   (emotional temperature)
      │
      ▼
Musical parameters
  tempo   = 60 + energy×60  BPM         (calm → energetic)
  mode    = major if warm+bright > 1.0  (happy vs. contemplative)
  root    = 110 × 2^(bright×2)  Hz      (dark → low pitch, bright → high)
  amp     = 0.25 + sat×0.35             (muted → quiet, vivid → loud)
      │
      ▼
Audio synthesis (pure numpy + stdlib wave)
  Layer 1: sustained drone pad  (root + 5th + octave)
  Layer 2: pentatonic arpeggio  (order seeded by hue+sat → deterministic)
  Layer 3: kick drum on downbeat of every measure
```

#### Example mappings

| Category | Expectation | Mode | Tempo |
|----------|-------------|------|-------|
| forest   | dark greens, low sat | minor | ~70 BPM |
| sea      | bright blues, med sat | major | ~85 BPM |
| street   | warm pavements, high energy | major | ~95 BPM |
| glacier  | cool pale blues, low energy | minor | ~65 BPM |

In [ ]:
spec = importlib.util.spec_from_file_location("s05", ROOT / "scripts" / "05_music_selector.py")
s05 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(s05)
s05.main()

In [ ]:
report = json.loads((ROOT / 'output' / 'music_selection_report.json').read_text())

rows = []
for cid, info in sorted(report.items()):
    cp = info['colour_profile']
    mp = info['music_params']
    rows.append({
        'Cluster': cid,
        'Dominant': info['dominant_category'],
        'Brightness': f"{cp['avg_brightness']:.2f}",
        'Saturation': f"{cp['avg_saturation']:.2f}",
        'Warmth':     f"{cp['warmth']:.2f}",
        'Mode':   mp['mode'],
        'Tempo':  f"{mp['tempo_bpm']} BPM",
        'Root':   f"{mp['root_hz']} Hz",
    })

pd.DataFrame(rows).set_index('Cluster')

In [ ]:
# Visualise colour profiles as a radar-style bar chart
props = ['avg_brightness', 'avg_saturation', 'warmth', 'energy']
labels_chart = ['Brightness', 'Saturation', 'Warmth', 'Energy']
x = np.arange(len(props))
width = 0.12

fig, ax = plt.subplots(figsize=(11, 5))
for i, (cid, info) in enumerate(sorted(report.items())):
    cp = info['colour_profile']
    vals = [cp[p] for p in props]
    dom  = info['dominant_category']
    ax.bar(x + i*width, vals, width, label=f'C{cid} ({dom})')

ax.set_xticks(x + width * (len(report)-1) / 2)
ax.set_xticklabels(labels_chart)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score (0–1)')
ax.set_title('Per-cluster colour profile driving music selection')
ax.legend(fontsize=8, ncol=2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

---
## Results & Discussion

### Clustering quality
KMeans on L2-normalised MobileNetV2 features achieves reasonable purity for an unsupervised method.  
The toughest confusions are **glacier ↔ sea** (both blue-dominant) and **buildings ↔ street** (both urban).  
A higher-dimensional clustering (e.g., UMAP + HDBSCAN) could separate these further.

### Feature extraction rationale
MobileNetV2's inverted-residual blocks learn to separate textures (fine detail), colours (tone), and shapes (scene structure).  
GlobalAveragePooling produces a translation-invariant summary — exactly what we need for scene similarity rather than object localisation.

### Music–visual alignment
The colour→music mapping captures intuitive correspondences:
- **Glacier / forest** (cool, dark, low saturation) → **minor mode, slow tempo** — evokes calm, introspective moods.
- **Sea / mountain** (bright, moderate saturation) → **major mode, medium tempo** — open, expansive feeling.
- **Street / buildings** (warm, higher energy) → **major mode, higher tempo** — urban activity.

The arpeggio note order is seeded by a hash of the cluster's colour statistics, ensuring the melody is uniquely determined by the visual content while avoiding random or manual selection.

### Generated videos
One final MP4 per cluster is available in `output/videos/cluster_<id>_final.mp4`.  
Each video contains ≤ 30 frames displayed for 3 s each with crossfade transitions and a custom synthesised soundtrack.

In [ ]:
final_videos = sorted((ROOT / 'output' / 'videos').glob('*_final.mp4'))
print('Final videos with music:')
for vf in final_videos:
    size_mb = vf.stat().st_size / 1e6
    print(f'  {vf.name:<38}  {size_mb:.1f} MB')